# FGSM move classifier: all characters, simulated match footage

Upload this file at colab.research.google.com (File → Upload notebook), then **Runtime → Change runtime type → T4 GPU**, and run the cells top to bottom.

Before you start, generate a download link for the code package on your Mac (valid 1 hour):
```
aws s3 presign s3://fgsm-vision-models-aibridix-official/fgsm_allchars_code.zip --expires-in 3600 --profile aibridix_official
```

**If Colab disconnects:** reconnect and run every cell again from the top. The dataset comes back from the Drive cache (no re-scrape) and training resumes from the last saved epoch.

In [ ]:
# 1. Check the GPU is attached - training on CPU would take days.
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then run this again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Google Drive holds the checkpoints and the dataset cache, so a disconnect doesn't lose them.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fgsm_runs /content/drive/MyDrive/fgsm_data

In [ ]:
# 3. Download the code + the 5-character model to start from. Paste the presigned URL when asked.
%cd /content
from getpass import getpass
code_url = getpass('Presigned S3 URL for fgsm_allchars_code.zip: ')
!wget -q -O fgsm_allchars_code.zip "{code_url}"
!rm -rf /content/fgsm_allchars && unzip -o -q fgsm_allchars_code.zip -d /content/fgsm_allchars
%cd /content/fgsm_allchars
!ls

In [ ]:
# 4. Install dependencies. torch/torchvision are skipped on purpose: Colab's own
# CUDA build must stay, and upgrading it from requirements.txt breaks the GPU.
!grep -vE '^(torch|torchvision)([=<>].*)?$' requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)

In [ ]:
# 5. (Optional) Back up checkpoints to S3 every 5 minutes as well as Drive.
# Skip this cell if you don't want to paste AWS keys - Drive alone is enough.
import getpass, os
os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('AWS Access Key ID: ')
os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('AWS Secret Access Key: ')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
!pip install -q awscli
get_ipython().system_raw(
    'while true; do sleep 300; '
    'aws s3 sync /content/drive/MyDrive/fgsm_runs/temporal_move_classifier_all_characters/ '
    's3://fgsm-vision-models-aibridix-official/colab_runs/temporal_move_classifier_all_characters/ --quiet; done &'
)
print('Background S3 sync started.')

In [ ]:
# 6. Dataset: every SF6 character's move clips from ultimateframedata.com.
# First run scrapes (~15 min, ~6-7 GB) and caches a tar on Drive; later runs restore it.
import os, shutil, subprocess
CACHE = '/content/drive/MyDrive/fgsm_data/ufd_all_characters.tar'
if os.path.exists(CACHE):
    print('Restoring dataset from', CACHE)
    subprocess.run(['tar', '-xf', CACHE, '-C', '/content/fgsm_allchars'], check=True)
else:
    subprocess.run(['python', 'src/scripts/scrape_ufd_gifs.py'], check=True)
    free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
    if free_gb > 12:
        subprocess.run(['tar', '-cf', CACHE, 'data/ufd'], check=True)
        print('Cached dataset to', CACHE)
    else:
        print(f'Only {free_gb:.1f} GB free on Drive - not caching; a reconnect will scrape again.')
!du -sh data/ufd
!echo "characters: $(ls data/ufd/move_classification_dataset | wc -l)"

In [ ]:
# 7. Train. Checkpoints go straight to Drive each epoch; re-running resumes from the latest.
#
# Colab's torch JIT-compiles a kernel on the first step and has crashed with
# "nvrtc: failed to open libnvrtc-builtins.so" when that library isn't on the
# loader path, so put it there first.
import glob, os, subprocess, sys

def _find_nvrtc_builtins():
    return glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/**/libnvrtc-builtins.so*', recursive=True)

hits = _find_nvrtc_builtins()
if not hits:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', 'nvidia-cuda-nvrtc==13.0.88'], check=False)
    hits = _find_nvrtc_builtins()
print('nvrtc builtins:', hits)
if hits:
    os.environ['LD_LIBRARY_PATH'] = os.path.dirname(hits[0]) + ':' + os.environ.get('LD_LIBRARY_PATH', '')

!sed -i 's#output_dir:.*#output_dir: /content/drive/MyDrive/fgsm_runs/temporal_move_classifier_all_characters#' configs/temporal_classifier_all_characters.yaml
!python src/train_temporal_classifier.py --config configs/temporal_classifier_all_characters.yaml

In [ ]:
# 8. Check the new model on the same probes as the 5-character model
# (runs/generalization/probes_base.json). Compare the "match-scale camera",
# "16 consecutive frames" and "all combined" rows - those collapsed before.
!python src/evaluate_generalization.py probes \
    --model-dir /content/drive/MyDrive/fgsm_runs/temporal_move_classifier_all_characters \
    --data-dir data/ufd \
    --json-out /content/drive/MyDrive/fgsm_runs/probes_all_characters.json